# DeFX Training Pipeline

This notebook walks through the complete DeFX training process:

1. **Setup** — Configure AWS, create S3 bucket and IAM role
2. **Data** — Download IDMT guitar dataset, upload to S3
3. **Ground Truth** — Generate dry/wet pairs using NAM amp models
4. **Train** — Fine-tune HDemucs on SageMaker
5. **Evaluate** — Download checkpoint and run inference

Create a `.env` file in the project root with your AWS config:
```
AWS_PROFILE=your-profile
AWS_REGION=us-east-1
```

## Configuration

In [ ]:
import subprocess, json, os, sys, time, zipfile, shutil
from pathlib import Path

# Load .env file
env_path = Path('.env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        if '=' in line and not line.startswith('#'):
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

AWS_PROFILE = os.environ.get('AWS_PROFILE', None)
AWS_REGION = os.environ.get('AWS_REGION', 'us-east-1')
BUCKET_PREFIX = 'defx'

# Training config
INSTANCE_TYPE = 'ml.g5.xlarge'
EPOCHS = 500
BATCH_SIZE = 4
LR = 2e-4
MAX_STEPS_PER_EPOCH = 200
UNFREEZE_ENCODER_LAYERS = 3
UNFREEZE_DECODER_LAYERS = 3

def aws(*args, capture=True):
    cmd = ['aws'] + list(args)
    if AWS_PROFILE:
        cmd += ['--profile', AWS_PROFILE]
    cmd += ['--region', AWS_REGION]
    r = subprocess.run(cmd, capture_output=capture, text=True)
    if r.returncode != 0 and capture:
        print(f'ERROR: {r.stderr}')
    return r

def aws_logs_cmd(prefix):
    cmd = f'aws logs tail /aws/sagemaker/TrainingJobs --follow --log-stream-name-prefix {prefix} --region {AWS_REGION}'
    if AWS_PROFILE:
        cmd += f' --profile {AWS_PROFILE}'
    return cmd

# Get account ID
r = aws('sts', 'get-caller-identity')
identity = json.loads(r.stdout)
ACCOUNT_ID = identity['Account']
BUCKET = f'{BUCKET_PREFIX}-{ACCOUNT_ID}'
ROLE_NAME = 'DefxSageMakerRole'
ROLE_ARN = f'arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}'

print(f'Region:   {AWS_REGION}')
print(f'Bucket:   {BUCKET}')
print(f'Role:     {ROLE_ARN}')

## 1. AWS Setup
Create S3 bucket and SageMaker execution role.

In [ ]:
# Create S3 bucket
r = aws('s3', 'ls', f's3://{BUCKET}')
if r.returncode != 0:
    print(f'Creating bucket: {BUCKET}')
    aws('s3', 'mb', f's3://{BUCKET}')
else:
    print(f'Bucket exists: {BUCKET}')

In [ ]:
# Create SageMaker execution role
trust_policy = json.dumps({
    'Version': '2012-10-17',
    'Statement': [{'Effect': 'Allow', 'Principal': {'Service': 'sagemaker.amazonaws.com'}, 'Action': 'sts:AssumeRole'}]
})

r = aws('iam', 'get-role', '--role-name', ROLE_NAME)
if r.returncode != 0:
    print(f'Creating role: {ROLE_NAME}')
    aws('iam', 'create-role', '--role-name', ROLE_NAME, '--assume-role-policy-document', trust_policy)
else:
    print(f'Role exists: {ROLE_NAME}')

role_policy = json.dumps({
    'Version': '2012-10-17',
    'Statement': [
        {'Effect': 'Allow', 'Action': ['s3:GetObject', 's3:PutObject', 's3:ListBucket', 's3:GetBucketLocation'],
         'Resource': [f'arn:aws:s3:::{BUCKET}', f'arn:aws:s3:::{BUCKET}/*',
                      f'arn:aws:s3:::sagemaker-{AWS_REGION}-{ACCOUNT_ID}', f'arn:aws:s3:::sagemaker-{AWS_REGION}-{ACCOUNT_ID}/*']},
        {'Effect': 'Allow', 'Action': ['logs:CreateLogGroup', 'logs:CreateLogStream', 'logs:PutLogEvents', 'logs:DescribeLogStreams'],
         'Resource': f'arn:aws:logs:*:{ACCOUNT_ID}:*'},
        {'Effect': 'Allow', 'Action': ['ecr:GetAuthorizationToken', 'ecr:BatchCheckLayerAvailability', 'ecr:GetDownloadUrlForLayer', 'ecr:BatchGetImage'],
         'Resource': '*'},
    ]
})
aws('iam', 'put-role-policy', '--role-name', ROLE_NAME, '--policy-name', 'DefxAccess', '--policy-document', role_policy)
print('Role permissions configured')

## 2. Download and Upload Training Data

In [ ]:
IDMT_URL = 'https://zenodo.org/records/7544110/files/IDMT-SMT-GUITAR_V2.zip?download=1'
IDMT_ZIP = Path('IDMT-SMT-GUITAR_V2.zip')
IDMT_DIR = Path('IDMT-SMT-GUITAR_V2')

# Check if already in S3
r = aws('s3', 'ls', f's3://{BUCKET}/idmt/')
if r.returncode == 0 and '.wav' in (r.stdout or ''):
    print('IDMT data already in S3, skipping download')
else:
    if not IDMT_DIR.exists():
        if not IDMT_ZIP.exists():
            print('Downloading IDMT dataset (~1.8GB)...')
            subprocess.run(['curl', '-L', '-o', str(IDMT_ZIP), IDMT_URL], check=True)
        print('Extracting...')
        with zipfile.ZipFile(IDMT_ZIP) as z:
            z.extractall('.')
    else:
        print(f'{IDMT_DIR} already exists')

    print('Uploading to S3...')
    aws('s3', 'sync', str(IDMT_DIR), f's3://{BUCKET}/idmt/',
        '--exclude', '*.xml', '--exclude', '*.pdf', '--exclude', '.DS_Store', capture=False)

    # Clean up local files
    print('Cleaning up local files...')
    if IDMT_ZIP.exists():
        IDMT_ZIP.unlink()
    if IDMT_DIR.exists():
        shutil.rmtree(IDMT_DIR)
    print('Done')

In [ ]:
# Upload NAM models
print('Uploading NAM models...')
aws('s3', 'sync', 'models/nam/blackpanel/', f's3://{BUCKET}/nam_models/', capture=False)
print('Done')

## 3. Generate Ground Truth
Processes each guitar recording through each NAM amp model. Results stream directly to S3.

In [ ]:
import boto3
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import SourceCode, InputData, Compute
from sagemaker.core.shapes.shapes import OutputDataConfig, StoppingCondition
from sagemaker.core.helper.session_helper import Session

boto_session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
sm_session = Session(boto_session=boto_session)
sm = boto_session.client('sagemaker')

# Set bucket in environment for the ground truth script
os.environ['DEFX_S3_BUCKET'] = BUCKET

CPU_IMAGE = f'763104351884.dkr.ecr.{AWS_REGION}.amazonaws.com/pytorch-training:2.5.1-cpu-py311-ubuntu22.04-sagemaker'

gt_trainer = ModelTrainer(
    training_image=CPU_IMAGE,
    source_code=SourceCode(source_dir='sagemaker', requirements='requirements.txt', entry_script='generate_ground_truth.py'),
    compute=Compute(instance_type='ml.m5.2xlarge', instance_count=1, volume_size_in_gb=100),
    role=ROLE_ARN,
    base_job_name='defx-ground-truth',
    sagemaker_session=sm_session,
    output_data_config=OutputDataConfig(s3_output_path=f's3://{BUCKET}/ground_truth'),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=28800),
    environment={'DEFX_S3_BUCKET': BUCKET},
)

gt_trainer.train(
    input_data_config=[
        InputData(channel_name='idmt', data_source=f's3://{BUCKET}/idmt'),
        InputData(channel_name='nam_models', data_source=f's3://{BUCKET}/nam_models'),
    ],
    wait=False,
)

gt_job = gt_trainer._latest_training_job.training_job_name
print(f'Ground truth job: {gt_job}')
print(f'Logs: {aws_logs_cmd(gt_job)}')

In [ ]:
# Wait for ground truth job
print('Waiting for ground truth generation...')
while True:
    desc = sm.describe_training_job(TrainingJobName=gt_job)
    status = desc['TrainingJobStatus']
    secondary = desc.get('SecondaryStatus', '')
    # Count files in S3
    s3 = boto_session.client('s3')
    try:
        wet_resp = s3.list_objects_v2(Bucket=BUCKET, Prefix='ground_truth/wet/', MaxKeys=1)
        wet_count = wet_resp.get('KeyCount', 0)
    except: wet_count = 0
    print(f'  {status} ({secondary}) — wet files in S3: {wet_count}+', end='\r')
    if status in ('Completed', 'Failed', 'Stopped'):
        break
    time.sleep(10)
print(f'\nJob finished: {status}')

## 4. Train
Fine-tune HDemucs on the ground truth data.

In [ ]:
from sagemaker.core.shapes.shapes import CheckpointConfig

GPU_IMAGE = f'763104351884.dkr.ecr.{AWS_REGION}.amazonaws.com/pytorch-training:2.5.1-gpu-py311-cu124-ubuntu22.04-sagemaker'

trainer = ModelTrainer(
    training_image=GPU_IMAGE,
    source_code=SourceCode(source_dir='sagemaker', requirements='requirements.txt', entry_script='train_demucs_defx.py'),
    compute=Compute(instance_type=INSTANCE_TYPE, instance_count=1, volume_size_in_gb=100),
    role=ROLE_ARN,
    base_job_name='defx-demucs',
    sagemaker_session=sm_session,
    output_data_config=OutputDataConfig(s3_output_path=f's3://{BUCKET}/output'),
    hyperparameters={
        'epochs': EPOCHS, 'batch-size': BATCH_SIZE, 'lr': LR,
        'chunk-samples': 44100, 'max-steps-per-epoch': MAX_STEPS_PER_EPOCH,
        'unfreeze-decoder-layers': UNFREEZE_DECODER_LAYERS,
        'unfreeze-encoder-layers': UNFREEZE_ENCODER_LAYERS,
        'save-every': 25,
    },
    stopping_condition=StoppingCondition(max_runtime_in_seconds=28800),
    checkpoint_config=CheckpointConfig(s3_uri=f's3://{BUCKET}/checkpoints'),
)

trainer.train(
    input_data_config=[
        InputData(channel_name='dry', data_source=f's3://{BUCKET}/ground_truth/dry'),
        InputData(channel_name='wet', data_source=f's3://{BUCKET}/ground_truth/wet'),
    ],
    wait=False,
)

train_job = trainer._latest_training_job.training_job_name
print(f'Training job: {train_job}')
print(f'Logs: {aws_logs_cmd(train_job)}')

In [ ]:
# Monitor training
import re
import matplotlib.pyplot as plt

logs_client = boto_session.client('logs')
LOG_GROUP = '/aws/sagemaker/TrainingJobs'

def get_log_lines(job):
    lines = []
    try:
        for stream in logs_client.describe_log_streams(logGroupName=LOG_GROUP, logStreamNamePrefix=job).get('logStreams', []):
            token = None
            while True:
                kw = {'logGroupName': LOG_GROUP, 'logStreamName': stream['logStreamName'], 'startFromHead': True}
                if token: kw['nextToken'] = token
                resp = logs_client.get_log_events(**kw)
                for e in resp['events']: lines.append(e['message'])
                nt = resp.get('nextForwardToken')
                if nt == token: break
                token = nt
    except: pass
    return lines

def parse_losses(lines):
    pat = re.compile(r'Epoch\s+(\d+)/(\d+)\s+loss=([\d.]+)(?:\s+l1=([\d.]+))?(?:\s+stft=([\d.]+))?(?:\s+mel=([\d.]+))?(?:\s+val=([\d.]+))?\s+time=([\d.]+)')
    return [{'epoch': int(m.group(1)), 'total': int(m.group(2)), 'loss': float(m.group(3)),
             'l1': float(m.group(4)) if m.group(4) else None, 'stft': float(m.group(5)) if m.group(5) else None,
             'mel': float(m.group(6)) if m.group(6) else None, 'val': float(m.group(7)) if m.group(7) else None,
             'time': float(m.group(8))} for line in lines if (m := pat.search(line))]

from IPython.display import clear_output
while True:
    desc = sm.describe_training_job(TrainingJobName=train_job)
    status = desc['TrainingJobStatus']
    losses = parse_losses(get_log_lines(train_job))
    clear_output(wait=True)
    if losses:
        ep = [d['epoch'] for d in losses]
        tl = [d['loss'] for d in losses]
        vl = [(d['epoch'], d['val']) for d in losses if d['val'] is not None]
        b = min(losses, key=lambda d: d['loss'])
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(ep, tl, 'b-', linewidth=1.5, label='Train')
        if vl: ax.plot([v[0] for v in vl], [v[1] for v in vl], 'r-', linewidth=1.5, label='Val')
        ax.axhline(y=b['loss'], color='b', linestyle='--', alpha=0.3)
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(); ax.grid(True, alpha=0.3)
        ax.set_title(f'Best: {b["loss"]:.4f} (epoch {b["epoch"]})')
        plt.tight_layout(); plt.show()
        avg_t = sum(d['time'] for d in losses) / len(losses)
        remaining = (losses[0]['total'] - ep[-1]) * avg_t / 3600
        print(f'Epoch {ep[-1]}/{losses[0]["total"]} | Best: {b["loss"]:.4f} | {avg_t:.0f}s/ep | ETA {remaining:.1f}h')
    else:
        print(f'{status} ({desc.get("SecondaryStatus", "")})')
    if status in ('Completed', 'Failed', 'Stopped'):
        print(f'Job finished: {status}'); break
    time.sleep(10)

## 5. Evaluate

In [ ]:
import tarfile

# Try model artifacts first (completed job), fall back to S3 checkpoints
desc = sm.describe_training_job(TrainingJobName=train_job)
ckpt_dir = Path('models/defx/checkpoints')
ckpt_dir.mkdir(parents=True, exist_ok=True)

if desc['TrainingJobStatus'] == 'Completed':
    model_s3 = desc['ModelArtifacts']['S3ModelArtifacts']
    print(f'Downloading model from {model_s3}')
    aws('s3', 'cp', model_s3, '/tmp/model.tar.gz', capture=False)
    with tarfile.open('/tmp/model.tar.gz') as t:
        t.extractall(ckpt_dir)
else:
    print(f'Job status: {desc["TrainingJobStatus"]} — downloading best checkpoint from S3')
    aws('s3', 'cp', f's3://{BUCKET}/checkpoints/demucs_defx_best.pt',
        str(ckpt_dir / 'demucs_defx_best.pt'), capture=False)

print(f'Checkpoints: {list(ckpt_dir.glob("*.pt"))}')

In [ ]:
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

TEST_INPUT = 'examples/test_wet.wav'
TEST_OUTPUT = 'examples/test_restored.wav'

if Path(TEST_INPUT).exists():
    !python inference.py --input {TEST_INPUT} --output {TEST_OUTPUT}
    print('\nRestored audio:')
    audio, sr = sf.read(TEST_OUTPUT)
    if audio.ndim > 1: audio = audio.mean(axis=1)
    display(Audio(audio, rate=sr))

    print('\nOriginal wet audio:')
    wet, sr = sf.read(TEST_INPUT)
    if wet.ndim > 1: wet = wet.mean(axis=1)
    display(Audio(wet, rate=sr))

    if Path('examples/test_dry.wav').exists():
        print('\nGround truth dry:')
        dry, sr = sf.read('examples/test_dry.wav')
        if dry.ndim > 1: dry = dry.mean(axis=1)
        display(Audio(dry, rate=sr))
else:
    print(f'Place a wet guitar WAV at {TEST_INPUT} and re-run this cell')
    print(f'Or run: python inference.py --input your_file.wav')